# IBES backfill — JOIN `statsumu_epsus` ⋈ `actpsumu_epsus`

Goal: assemble, per `(TICKER, STATPERS)`, the **forward (FY2) consensus EPS** from the
estimates file and the **contemporaneous PRICE** from the actuals+price file, so we can
compute an IBES forward P/E that later gets interpolated to daily using yfinance prices.

- **`statsumu_epsus`** — estimates. Many rows per `(TICKER, STATPERS)`: one per `FPI`
  (1=current FY, 2=next FY...), `FISCALP` (ANN/QTR), and sometimes per `CURCODE`.
- **`actpsumu_epsus`** — one row per `(TICKER, STATPERS)`: `PRICE`/`PRDAYS`/`CURR_PRICE`
  plus realized actuals `FY0A` (last annual) and `INT0A` (last interim).

Join key: **`(TICKER, STATPERS)`**. We slice the estimates to `FISCALP=ANN`, `FPI=2`
(the next-FY consensus = what Yahoo's forward P/E uses) before joining.

> These CSVs are ~1 GB / ~1.6 GB. We filter to the watchlist's IBES codes *while reading*
> in chunks, so nothing huge lands in memory. pandas' CSV parser respects quoting, so the
> comma-in-CNAME corruption we saw with awk does **not** happen here.


In [ ]:
import tomllib
from pathlib import Path
import pandas as pd

PE_DIR = Path.cwd()
if not (PE_DIR / "ibes.statsumu_epsus.csv").exists():
    PE_DIR = Path("/root/repos/market-monitors/pe_monitor")

STAT_CSV = PE_DIR / "ibes.statsumu_epsus.csv"
ACT_CSV  = PE_DIR / "ibes.actpsumu_epsus.csv"

# Watchlist IBES proprietary codes, straight from config.toml [ibes.ticker_map].
with open(PE_DIR / "config.toml", "rb") as f:
    cfg = tomllib.load(f)
ticker_map = cfg["ibes"]["ticker_map"]          # yahoo_ticker -> IBES code
code_to_yahoo = {v: k for k, v in ticker_map.items()}
CODES = set(ticker_map.values())
print(f"{len(CODES)} IBES codes from config:")
print(sorted(CODES))


## Chunked, filtered loader

Reads only rows whose `TICKER` is in the watchlist. Everything as `str` first (no dtype
guessing); we coerce numerics explicitly afterwards.


In [ ]:
def load_ibes(path, codes, chunksize=1_000_000):
    """Stream a big IBES CSV, keeping only rows whose TICKER is in `codes`."""
    keep = []
    n_read = 0
    for chunk in pd.read_csv(path, dtype=str, chunksize=chunksize,
                             na_filter=False, low_memory=False):
        n_read += len(chunk)
        keep.append(chunk[chunk["TICKER"].isin(codes)])
    df = pd.concat(keep, ignore_index=True)
    print(f"{path.name}: scanned {n_read:,} rows -> kept {len(df):,}")
    return df

df_stat = load_ibes(STAT_CSV, CODES)   # ~30-90s
df_stat.head()


In [ ]:
df_act = load_ibes(ACT_CSV, CODES)     # ~30-90s
df_act.head()


## Inspect the estimates file (`statsumu_epsus`)

Look at how many rows exist per `FPI` / `FISCALP` / `CURCODE`, and which watchlist names
were actually found.


In [ ]:
print("FISCALP:\n", df_stat["FISCALP"].value_counts(), "\n")
print("FPI:\n", df_stat["FPI"].value_counts(), "\n")
print("CURCODE:\n", df_stat["CURCODE"].value_counts(), "\n")
print("MEASURE:\n", df_stat["MEASURE"].value_counts(), "\n")

# coverage: which watchlist codes are present / missing
found = set(df_stat["TICKER"].unique())
print("MISSING from statsumu_epsus:", sorted(CODES - found))
cov = (df_stat.groupby("TICKER")["STATPERS"]
       .agg(["count", "min", "max"]).rename_axis("ibes_code"))
cov.insert(0, "yahoo", cov.index.map(code_to_yahoo))
cov.sort_values("yahoo")


## Inspect the price+actuals file (`actpsumu_epsus`)

Confirm `PRDAYS` ≈ `STATPERS` (price taken contemporaneously), and look at the currency
fields `CURCODE` (actual EPS currency) vs `CURR_PRICE` (price currency).


In [ ]:
print("columns:", list(df_act.columns), "\n")
print("CURCODE (actual):\n", df_act["CURCODE"].value_counts(), "\n")
print("CURR_PRICE (price):\n", df_act["CURR_PRICE"].value_counts(), "\n")

# Does PRDAYS line up with STATPERS? (same monthly Thursday cut)
tmp = df_act[["TICKER","STATPERS","PRDAYS","PRICE","CURR_PRICE","FY0A","INT0A","FY0EDATS"]].copy()
tmp["lag_days"] = (pd.to_datetime(tmp["STATPERS"]) - pd.to_datetime(tmp["PRDAYS"], errors="coerce")).dt.days
print("STATPERS - PRDAYS lag (days) distribution:\n", tmp["lag_days"].describe())
tmp.head(10)


## Build the FY2 estimate slice and JOIN

1. Keep `FISCALP=ANN`, `FPI=2` from estimates (the next-FY consensus).
2. Join to the price/actuals row on `(TICKER, STATPERS)`.
3. Compute `fwd_pe = PRICE / MEANEST` and flag whether estimate currency == price currency
   (the P/E is only meaningful when they match).


In [ ]:
# 1) next-FY annual estimate
fy2 = df_stat[(df_stat["FISCALP"] == "ANN") & (df_stat["FPI"] == "2")].copy()
for c in ["MEANEST","MEDEST","NUMEST"]:
    fy2[c] = pd.to_numeric(fy2[c], errors="coerce")
fy2 = fy2[["TICKER","STATPERS","CURCODE","FPEDATS","MEANEST","MEDEST","NUMEST"]]
fy2 = fy2.rename(columns={"CURCODE":"est_curcode", "FPEDATS":"fy2_pends"})

# 2) price/actuals side
act = df_act[["TICKER","STATPERS","PRICE","PRDAYS","CURR_PRICE","CURCODE",
              "FY0A","INT0A","FY0EDATS","INT0DATS","SHOUT"]].copy()
for c in ["PRICE","FY0A","INT0A","SHOUT"]:
    act[c] = pd.to_numeric(act[c], errors="coerce")
act = act.rename(columns={"CURCODE":"act_curcode"})

# 3) join
j = fy2.merge(act, on=["TICKER","STATPERS"], how="inner")
j["currency_match"] = j["est_curcode"] == j["CURR_PRICE"]
j["fwd_pe"] = j["PRICE"] / j["MEANEST"]
j["yahoo"] = j["TICKER"].map(code_to_yahoo)
j["STATPERS"] = pd.to_datetime(j["STATPERS"])
j = j.sort_values(["yahoo","STATPERS"]).reset_index(drop=True)

print(f"{len(j):,} joined (TICKER, STATPERS) FY2 rows")
print("currency_match:\n", j["currency_match"].value_counts())
print("\nrows whose est currency != price currency (need care):")
print(j[~j["currency_match"]].groupby(["yahoo","est_curcode","CURR_PRICE"]).size())
j.head(20)


## Per-ticker time series

Pick one name and eyeball the monthly anchor series: price, next-FY EPS, forward P/E.
This is exactly the monthly anchor we'll later interpolate to daily with yfinance prices
via `pe_ibes(d) = (PRICE/MEANEST) * price_yf(d)/price_yf(STATPERS)`.


In [ ]:
SYM = "NVDA"   # try "0700.HK", "BABA", "005930.KS", ...
code_ = ticker_map[SYM]
view = j[j["TICKER"] == code_][
    ["STATPERS","fy2_pends","MEANEST","PRICE","CURR_PRICE","est_curcode",
     "currency_match","fwd_pe","FY0A","INT0A","NUMEST"]
]
print(f"{SYM} ({code_}) — {len(view)} monthly anchors")
view.tail(24)


### Notes / caveats to keep in mind while examining

- **Currency match.** Where `currency_match` is False (e.g. EPS in CNY but `CURR_PRICE`
  HKD), `fwd_pe` is *not* a real P/E. Either pick the matching `CURCODE` cut from the
  estimates file, or rely on the dimensionless anchor+ratio trick downstream.
- **Cadence is monthly** (one `STATPERS` ≈ third Thursday). Daily resolution comes later
  from yfinance price ratios, not from IBES.
- **FY2 = Yahoo's convention** (next full fiscal year), so this overlays Yahoo's forward
  P/E directly; expect a small step once a year at the fiscal roll.
- **Actuals** (`FY0A`/`INT0A`) are IBES *street* EPS — fine for an internally consistent
  IBES trailing P/E, but won't equal Yahoo's GAAP `trailingEps`.
